# ಪಾಠ 03 - ಏಜೆಂಟಿಕ್ ವಿನ್ಯಾಸ ಮಾದರಿಗಳು

ಈ ಪಾಠದಲ್ಲಿ, ಪರಿಣಾಮಕಾರಿ AI ಏಜೆಂಟ್ಗಳನ್ನು ನಿರ್ಮಿಸುವ ಮೂವರು ಮೂಲಭೂತ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಅನ್ವೇಷಣೆ ಮಾಡುತ್ತೇವೆ:

1. **ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು** — ಏಜೆಂಟ್ ವರ್ತನೆಯನ್ನು ಮಾರ್ಗದರ್ಶಿಸುವ ನಿಖರ, ಪಾತ್ರವನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುವ ಪ್ರಾಂಪ್ಟ್‌ಗಳನ್ನು ರಚಿಸುವುದು
2. **Pydantic ಮಾದರಿಗಳೊಂದಿಗೆ ಸಂರಚಿತ ಔಟ್ಪುಟ್** — ಏಜೆಂಟ್‌ಗಳು ನಿರೀಕ್ಷಿತ, ಮಾನ್ಯತೆಯನ್ನೂ ಪಡೆದ ಡೇಟಾವನ್ನು ಹಿಂತಿರುಗಿಸುವುದನ್ನು ಖಚಿತಪಡಿಸುವುದು
3. **ಒಂದು ಜವಾಬ್ದಾರಿ ಏಜೆಂಟ್‌ಗಳು** — ಒಟ್ಟಾಗಿ ಒಬ್ಬೊಬ್ಬರು ಒಂದು ಕಾರ್ಯವನ್ನು ಚೆನ್ನಾಗಿ ನಿರ್ವಹಿಸುವ ದೃಷ್ಟಿಕೋನದ ಏಜೆಂಟ್‌ಗಳನ್ನು ವಿನ್ಯಾಸಗೊಳಿಸುವುದು

ನಾವು ಪ್ರತಿಯೊಂದು ಮಾದರಿಯನ್ನೂ **ಪ್ರಯಾಣ ಗುರಿ ಶಿಫಾರಸು ಮಾಡುವವರ** ದೃಶ್ಯರೇಖೆಗೆ ಅನ್ವಯಿಸಿ, ಕ್ರಮೇಣ ಗುರಿಗಳನ್ನು ಸಲಹೆ ಮಾಡಲು, ಲಭ್ಯತೆಯನ್ನು ಪರಿಶೀಲಿಸಲು ಮತ್ತು ಲಾಜಿಸ್ಟಿಕ್ಸ್ ಅನ್ನು ಹ್ಯಾಂಡಲ್ ಮಾಡಲು ಸಾಧ್ಯವಿರುವ ವ್ಯವಸ್ಥೆಯನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.


## ಸಂಯೋಜನೆ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ಪ್ಯಾಟರ್ನ್ 1: ಸ್ಪಷ್ಟ ಏಜೆಂಟ್ ಸೂಚನೆಗಳು

ಅತ್ಯಂತ ಪರಿಣಾಮಕಾರಿ ಪ್ಯಾಟರ್ನ್ ಸಹ ಅದೇ ಆಗಿದೆ: ನಿಮ್ಮ ಏಜೆಂಟ್‌ಗೆ ಸ್ಪಷ್ಟ, ವಿವರವಾದ ಸೂಚನೆಗಳನ್ನು ಬರೆಯುವುದು.

ಉತ್ತಮ ಸೂಚನೆಗಳು ನಿರ್ದೇಶಿಸುತ್ತವೆ:
- **ಯಾರು** ಏಜೆಂಟ್ ಆಗಿದ್ದಾರೆ (ಪರ್ಸೋನ ಮತ್ತು ಶೈಲಿ)
- **ಏನು** ಅದು ಮಾಡಬಹುದೆಂದು (ಹಂತ ಹಂತದ ಜವಾಬ್ದಾರಿಗಳು)
- **ಹೇಗೆ** ಅದು ವರ್ತಿಸಬೇಕು (ನಿಬಂಧನೆಗಳು ಮತ್ತು ಶೈಲಿ)

ಕೆಳಗಿನ ಉದಾಹರಣೆಯಲ್ಲಿ, ನಾವು ಪ್ರತಿಯೊಂದು ಪ್ರತಿಕ್ರಿಯೆಯನ್ನೂ ರೂಪಿಸುವ ಸ್ಪಷ್ಟ ಸೂಚನೆಗಳೊಂದಿಗೆ ಒಂದು ಪ್ರವಾಸಿ_concierge ಏಜೆಂಟ್ ಸೃಷ್ಟಿಸುತ್ತೇವೆ.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## ಮಾದರಿ 2: ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳೊಂದಿಗೆ ರಚಿಸಲಾದ_Output

ಓಪನ್-ಆಕಾರದ ಪಠ್ಯ ಸಂಭಾಷಣೆಗೆ ಉಪಯುಕ್ತವಾಗಿದ್ದರೂ, ಕೆಳಗಿನ ವ್ಯವಸ್ಥೆಗಳು ರಚಿಸಲಾದ ಡೇಟಾವನ್ನು ಬೇಕು.
**ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳು** ಮತ್ತು **ಟೂಲ್ ಫಂಕ್ಷನ್** ಜೋಡಿಸುವ ಮೂಲಕ, ನಾವು:

- ಏಜೆಂಟ್‌ನ ಔಟ್‌ಪುಟ್‌ಗೆ ನಿಖರವಾದ ಸ್ಕೀಮಾ ನಿರ್ಧರಿಸಬಹುದು
- ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಮಾನ್ಯತೆ ಮಾಡಬಹುದು
- ಏಜೆಂಟ್ ಫಲಿತಾಂಶಗಳನ್ನು ಅನ್ವಯಿಕ ಲಾಜಿಕ್ಗೆ ವಿಶ್ವಾಸಾರ್ಹವಾಗಿ ಸಂಯೋಜಿಸಬಹುದಾಗಿದೆ

ನಾವು ಒಂದು ಟೂಲ್ ಅನ್ನು ಕೂಡ ಪರಿಚಯಿಸುತ್ತೇವೆ ಅದು ಗಮ್ಯದ ವಿವರಗಳನ್ನು ಸರಬರಾಜು ಮಾಡುತ್ತದೆ ಹಾಗಾಗಿ ಏಜೆಂಟ್ ತನ್ನ ಶಿಫಾರಸುಗಳನ್ನು ನಿಜವಾದ ಡೇಟಾದಲ್ಲಿ ನೆರೆದಿಡುತ್ತದೆ.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## ಮಾದರಿ 3: ಏಕ ಜವಾಬ್ದಾರಿ ಏಜೆಂಟುಗಳು

ಸಂಕೀರ್ಣ ಕಾರ್ಯಗಳು ಹಲವಾರು ಧ್ಯೇಯ ಹೊಂದಿದ ಏಜೆಂಟ್‌ಗಳ ಮೇಲೆ ಕೆಲಸವನ್ನು ವಿಭಜಿಸುವುದರಿಂದ ಲಾಭವಾಗುತ್ತದೆ, ಪ್ರತಿ ಏಜೆಂಟ್ ಗೆ ಒಂದು ಜವಾಬ್ದಾರಿ ಇರುತ್ತದೆ:

- ಸ್ಥಳಗಳು ಮತ್ತು ಲಭ್ಯತೆಗಳಿಗೆ ಸಂಬಂಧಿಸಿದ **ಗಮ್ಯಸ್ಥಳ ಪರಿಣತಿ**
- ವಿಮಾನಗಳ, ಹೋಟೆಲ್‌ಗಳ ಮತ್ತು ಪ್ರಯಾಣಯೋಜನೆಗಳನ್ನು ನಿರ್ವಹಿಸುವ **ಲಾಜಿಸ್ಟಿಕ್ ಪ್ಲ್ಯಾನರ್**

ಇದು ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರಿಂಗ್ ತತ್ವವಾದ *ಪರಾಕಾಷ್ಠೆಯ ವಿಭಜನೆ*ಯನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ — ಪ್ರತಿ ಏಜೆಂಟ್ ಅನ್ನು ಸ್ವತಂತ್ರವಾಗಿ ಪರೀಕ್ಷಿಸಲು, ನಿರ್ವಹಿಸಲು ಮತ್ತು ಸುಧಾರಿಸಲು ಸುಲಭ.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನಾವು ಪ್ರಯಾಣ ಶಿಫಾರಸು ಮಾಡುವ ದೃಶ್ಯದೊಂದಿಗೆ ಮೂರು ಪ್ರತಿನಿಧಿ ವಿನ್ಯಾಸ ಮಾದರಿಗಳನ್ನು ಅನ್ವಯಿಸಿದ್ದೇವೆ:

| ಮಾದರಿ | ಮುಖ್ಯ ಆಲೋಚನೆ | ಪ್ರಯೋಜನ |
|---|---|---|
| **ಸ್ಪಷ್ಟ ನಿರ್ದೇಶನಗಳು** | ವ್ಯಕ್ತಿತ್ವ, ಕರ್ತವ್ಯಗಳು, ಮತ್ತು ನಿರ್ಬಂಧಗಳನ್ನು ಮೊದಲೇ ನಿರ್ಧರಿಸಿ | ಸತತ, ಬ್ರ್ಯಾಂಡ್‌ಗೆ ಹೊಂದಿಕೆಯಾಗುವ ಪ್ರತಿನಿಧಿ ವರ್ತನೆ |
| **ರಚನಾತ್ಮಕ ಉತ್ಪನ್ನ** | ಪ್ರತಿಕ್ರಿಯಾ ಮಾದರಿಯಾಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ಬಳಸಿ | ಪರಿಶೀಲಿತ, ಯಂತ್ರ ಓದುವಾಯಕ ಫಲಿತಾಂಶಗಳು |
| **ಒಂದು ಕర్తವ್ಯದ ಹೊಣೆಗಾರಿಕೆ** | ಪ್ರತಿಯೊಂದು ಪ್ರತಿನಿಧಿಗೆ ಒಂದು ಹೆಚ್ಚುವರಿ ಕೆಲಸವನ್ನೇ ನೀಡಿ | ಪರೀಕ್ಷೆ ಮಾಡುವುದು, ನಿರ್ವಹಿಸುವುದು, ಮತ್ತು ಸಂಯೋಜಿಸುವುದು ಸುಲಭ |

ಈ ಮಾದರಿಗಳು ಸಹಜವಾಗಿ ಸಂಯೋಜನೆಯಾಗುತ್ತವೆ — ನೀವು ಸ್ಪಷ್ಟ ನಿರ್ದೇಶನಗಳನ್ನು ರಚನಾತ್ಮಕ ಉತ್ಪನ್ನದೊಂದಿಗೆ ಒಂದೇ ಕर्तವ್ಯದ ಹೊಣೆಗಾರಿಕೆಯುಳ್ಳ ಪ್ರತಿನಿಧಿಯಲ್ಲಿ ಸೇರಿಸಿ ಬಲಿಷ್ಠ, ಉತ್ಪಾದನಾ ಸಿದ್ಧ ವ್ಯವಸ್ಥೆಗಳನ್ನು ನಿರ್ಮಿಸಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಎಚ್ಚರಿಕೆ**:
ಈ ದಸ್ತಾವೇಜು [Co-op Translator](https://github.com/Azure/co-op-translator) ಎಂಬ AI ಅನುವಾದ ಸೇವೆಯನ್ನು ಬಳಸಿಕೊಂಡು ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿರೋದಾದರೂ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿಂದಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ವಿಚಿತ್ರತೆಗಳು ಇರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜುವೇ ಅಧಿಕೃತ ಉಲ್ಲೇಖ ಎಂದು ಪರಿಗಣಿಸಬೇಕಾಗಿದೆ. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನೇ ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದ ಬಳಕೆಯಿಂದ ಉಂಟಾಗುವ ಯಾವ ಅಭಿಪ್ರಾಯ ಭೇದ ಅಥವಾ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೆಗಾಗಿ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
